# Notebook 2 — Feature Engineering
## ImmoPredict SN — Construction des Variables
---

In [1]:
import pandas as pd
import numpy as np
import math
import re
import json
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

GOLD = '#C9A84C'; NAVY = '#0F2444'

# Charger les données nettoyées du NB1
df = pd.read_csv('../data/dataset_vente.csv')
print(f'Dataset vente chargé: {len(df):,} lignes, {df.shape[1]} colonnes')
df.head(3)

Dataset vente chargé: 2,768 lignes, 17 colonnes


,id,url,title,price,surface_area,bedrooms,bathrooms,city,description,latitude,longitude,scraped_at,statut,property_type,source,transaction,property_type_clean
0,cf884190d8da48c63d445efd60fb11fc,https://sn.coinafrique.com/annonce/villas/vent...,Vente villa,60000000,NaN,NaN,NaN,Mbao,"{""Maison inachevé niveau 1 à petit mbao\ntitre...",14.729935,-17.325860,2026-03-24 03:28:07.479436,Particulier,"Mbao, Dakar, Sénégal",coinafrique,Vente,Autre
1,792792f244f2daad8d32e80d78a0bcca,https://sn.coinafrique.com/annonce/terrains/te...,Terrain 150,30000000,150.0,NaN,NaN,Mbao,"{""VENDRE TERRAIN 150 m² | MBAO VILLENEUVE\nTer...",14.729935,-17.325860,2026-03-24 02:00:37.596492,Pro,"Mbao, Dakar, Sénégal",coinafrique,Vente,Autre
2,0e7bb7d24a5a523e06379be8c88b3a87,https://sn.coinafrique.com/annonce/terrains/te...,Terrains 221,100000000,221.0,NaN,NaN,Hlm,"{""Terrain à vendre à hlm grand médine\r\nsuper...",14.700117,-17.445505,2026-03-24 02:00:34.139954,Pro,"HLM, Dakar, Sénégal",coinafrique,Vente,Autre


In [2]:
# ══════════════════════════════════════════════════════════════
# 1. COORDONNÉES GPS PAR QUARTIER
# ══════════════════════════════════════════════════════════════
CITY_GPS = {
    # Dakar centre et côte ouest (premium)
    'almadies':           (14.745, -17.510),
    'ngor':               (14.749, -17.514),
    'yoff':               (14.758, -17.490),
    'ouakam':             (14.724, -17.494),
    'mermoz':             (14.710, -17.475),
    'plateau':            (14.693, -17.447),
    'fann':               (14.696, -17.460),
    'sacre-coeur':        (14.720, -17.461),
    'sacre coeur':        (14.720, -17.461),
    'vdn':                (14.730, -17.470),
    'point e':            (14.694, -17.460),
    'corniche':           (14.710, -17.470),
    # Dakar intermédiaire
    'sicap':              (14.712, -17.462),
    'liberte':            (14.715, -17.463),
    'hlm':                (14.713, -17.459),
    'medina':             (14.695, -17.456),
    'grand yoff':         (14.736, -17.467),
    'dieuppeul':          (14.714, -17.457),
    'patte d\'oie':       (14.725, -17.460),
    'nord foire':         (14.742, -17.465),
    'parcelles':          (14.748, -17.451),
    'parcelles assainies':(14.748, -17.451),
    # Banlieue
    'pikine':             (14.755, -17.395),
    'guediawaye':         (14.778, -17.393),
    'thiaroye':           (14.755, -17.370),
    'yeumbeul':           (14.765, -17.348),
    'keur massar':        (14.765, -17.340),
    'mbao':               (14.740, -17.320),
    'rufisque':           (14.716, -17.274),
    'bargny':             (14.696, -17.239),
    # Autres villes
    'dakar':              (14.693, -17.447),
    'thies':              (14.791, -16.926),
    'mbour':              (14.368, -16.965),
    'saly':               (14.454, -17.012),
    'diamniadio':         (14.727, -17.184),
    'kaolack':            (14.150, -16.083),
    'saint-louis':        (16.018, -16.490),
    'ziguinchor':         (12.583, -16.267),
    'default':            (14.693, -17.447),  # Dakar par défaut
}

def get_gps(city):
    if not city or pd.isna(city): return CITY_GPS['default']
    k = city.lower().strip()
    if k in CITY_GPS: return CITY_GPS[k]
    # Recherche partielle
    for key, coords in CITY_GPS.items():
        if key in k or k in key: return coords
    return CITY_GPS['default']

df['gps'] = df['city'].apply(get_gps)
df['lat'] = df['gps'].apply(lambda x: x[0])
df['lon'] = df['gps'].apply(lambda x: x[1])
df.drop('gps', axis=1, inplace=True)

# Pour les annonces avec GPS réel, utiliser les vraies coordonnées
if 'latitude' in df.columns:
    mask = df['latitude'].notna() & df['latitude'].between(12, 17)
    df.loc[mask, 'lat'] = df.loc[mask, 'latitude']
    df.loc[mask, 'lon'] = df.loc[mask, 'longitude']
    print(f'GPS réel utilisé pour {mask.sum()} annonces sur {len(df)}')

print('✅ GPS assignés')

GPS réel utilisé pour 1252 annonces sur 2768
✅ GPS assignés


In [3]:
# ══════════════════════════════════════════════════════════════
# 2. DISTANCES AUX POINTS D'INTÉRÊT (POI)
# ══════════════════════════════════════════════════════════════
POI = {
    'dist_mer':      (14.693, -17.459),   # Corniche
    'dist_centre':   (14.693, -17.447),   # Plateau (CBD)
    'dist_aeroport': (14.741, -17.490),   # Aéroport LSS
    'dist_aibd':     (14.738, -17.091),   # Aéroport AIBD (Diass)
    'dist_port':     (14.672, -17.427),   # Port de Dakar
    'dist_ucad':     (14.692, -17.464),   # Université UCAD
    'dist_vdn':      (14.730, -17.470),   # VDN (axe routier)
    'dist_corniche': (14.710, -17.470),   # Corniche Ouest
    'dist_parc':     (14.700, -17.458),   # Parc de Hann
}

def haversine(lat1, lon1, lat2, lon2):
    """Distance en km entre deux points GPS."""
    R = 6371.0
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlambda = math.radians(lon2 - lon1)
    a = math.sin(dphi/2)**2 + math.cos(phi1)*math.cos(phi2)*math.sin(dlambda/2)**2
    return R * 2 * math.atan2(math.sqrt(a), math.sqrt(1-a))

for col, (lat2, lon2) in POI.items():
    df[col] = df.apply(lambda r: haversine(r['lat'], r['lon'], lat2, lon2), axis=1)

print('✅ Distances POI calculées')
print(df[[c for c in df.columns if c.startswith('dist_')]].describe().round(2))

✅ Distances POI calculées
       dist_mer  dist_centre  dist_aeroport  dist_aibd  dist_port  dist_ucad  \
count   2768.00      2768.00        2768.00    2768.00    2768.00    2768.00   
mean      18.10        17.46          21.02      39.14      19.00      18.43   
std       31.88        31.72          32.47      18.85      29.96      31.92   
min        0.15         0.00           0.68      10.08       1.51       0.48   
25%        1.29         0.00           4.49      37.73       3.18       1.83   
50%        3.23         3.62           7.06      38.61       6.72       3.31   
75%       19.71        18.49          22.04      40.58      17.47      20.25   
max      269.65       269.03         275.93     257.71     265.95     269.81   

       dist_vdn  dist_corniche  dist_parc  
count   2768.00        2768.00    2768.00  
mean      19.35          18.67      17.98  
std       32.20          32.25      31.96  
min        0.00           0.00       0.44  
25%        4.64           3.11   

In [4]:
# ══════════════════════════════════════════════════════════════
# 3. FEATURES GÉOGRAPHIQUES DÉRIVÉES
# ══════════════════════════════════════════════════════════════
PREMIUM_ZONES = {
    'almadies', 'ngor', 'mermoz', 'fann', 'plateau', 'sacre-coeur',
    'sacre coeur', 'point e', 'corniche', 'vdn', 'yoff'
}

df['is_premium'] = df['city'].str.lower().apply(
    lambda c: int(any(z in str(c) for z in PREMIUM_ZONES)) if pd.notna(c) else 0
)

# Score de zone premium basé sur distance mer (inversement proportionnel)
df['zone_premium'] = (10 - df['dist_mer'].clip(0, 10)).round(2)

# Log des distances (réduire la skewness)
for col in [c for c in df.columns if c.startswith('dist_')]:
    df[f'log_{col}'] = np.log1p(df[col])

print('Features géographiques créées')
print(f'Biens en zone premium : {df["is_premium"].sum():,} ({df["is_premium"].mean()*100:.1f}%)')

Features géographiques créées
Biens en zone premium : 309 (11.2%)


In [5]:
# ══════════════════════════════════════════════════════════════
# 4. FEATURES NLP (description + titre)
# ══════════════════════════════════════════════════════════════
NLP_FEATURES = {
    'has_standing':      ['standing', 'grand standing', 'luxe', 'haut standing', 'premium'],
    'has_neuf':          ['neuf', 'nouvelle construction', 'tout neuf', 'livraison'],
    'has_renove':        ['renov', 'refait', 'refection', 'modernise', 'retape'],
    'has_piscine':       ['piscine', 'pool', 'swimming'],
    'has_meuble':        ['meuble', 'equipe', 'amenage', 'all inclusive'],
    'has_climatise':     ['clim', 'climatise', 'climatisation', 'air conditionne'],
    'has_ascenseur':     ['ascenseur', 'elevator', 'lift'],
    'has_cuisine_amer':  ['cuisine americaine', 'cuisine integree', 'open kitchen', 'americaine'],
    'has_parking':       ['parking', 'garage', 'carport'],
    'has_jardin':        ['jardin', 'garden', 'verdure', 'espace vert'],
    'has_balcon':        ['balcon', 'terrasse', 'loggia', 'veranda'],
    'has_gardiennage':   ['gardien', 'vigile', 'securite', 'gardiennage', 'surveillance'],
    'has_groupe_elec':   ['groupe electrogene', 'generateur', 'groupe elec', 'gen set'],
    'has_vue_mer':       ['vue mer', 'face mer', 'bord de mer', 'ocean', 'vue sur mer'],
    'has_concierge':     ['concierge', 'reception', 'accueil'],
    'has_digicode':      ['digicode', 'interphone', 'visiophone', 'badge', 'code acces'],
    'has_titre_foncier': ['titre foncier', 'tf', 'foncier regularise', 'regularise'],
    'has_viabilise':     ['viabilise', 'eau electricite', 'borne fontaine', 'vrd'],
    'has_invest':        ['investissement', 'rendement', 'rentable', 'locatif', 'revenu'],
    'has_vue_ville':     ['vue ville', 'panoramique', 'vue sur dakar'],
}

def text_features(row):
    text = (str(row.get('description') or '') + ' ' + str(row.get('title') or '')).lower()
    return {feat: int(any(kw in text for kw in kws)) for feat, kws in NLP_FEATURES.items()}

nlp_df = df.apply(lambda row: pd.Series(text_features(row)), axis=1)
df = pd.concat([df, nlp_df], axis=1)

# Score de prestige
prestige_cols = ['has_standing', 'has_piscine', 'has_vue_mer', 'has_gardiennage',
                 'has_climatise', 'has_titre_foncier', 'has_ascenseur']
df['n_premium_feats'] = df[prestige_cols].sum(axis=1)

print('Features NLP créées')
print('\nPrévalence des équipements :')
for col in list(NLP_FEATURES.keys())[:10]:
    print(f'  {col}: {df[col].mean()*100:.1f}%')

Features NLP créées

Prévalence des équipements :
  has_standing: 3.5%
  has_neuf: 2.6%
  has_renove: 0.1%
  has_piscine: 4.4%
  has_meuble: 6.2%
  has_climatise: 1.2%
  has_ascenseur: 2.4%
  has_cuisine_amer: 0.0%
  has_parking: 6.7%
  has_jardin: 2.6%


In [6]:
# ══════════════════════════════════════════════════════════════
# 5. FEATURES NUMÉRIQUES DÉRIVÉES
# ══════════════════════════════════════════════════════════════

# Valider surface_area
df['surface_area'] = pd.to_numeric(df['surface_area'], errors='coerce')
df.loc[df['surface_area'] <= 0, 'surface_area'] = np.nan
df.loc[df['surface_area'] > 50000, 'surface_area'] = np.nan  # absurde

# Valider chambres
df['bedrooms'] = pd.to_numeric(df.get('bedrooms'), errors='coerce')
df.loc[df['bedrooms'] > 30, 'bedrooms'] = np.nan

df['bathrooms'] = pd.to_numeric(df.get('bathrooms'), errors='coerce')
df.loc[df['bathrooms'] > 20, 'bathrooms'] = np.nan

# Features dérivées
df['bedrooms_fill']  = df['bedrooms'].fillna(df['bedrooms'].median())
df['bathrooms_fill'] = df['bathrooms'].fillna(df['bathrooms'].median())
df['surface_fill']   = df['surface_area'].fillna(df['surface_area'].median())

df['rooms_total']      = df['bedrooms_fill'] + df['bathrooms_fill']
df['surface_per_room'] = df['surface_fill']  / df['rooms_total'].clip(1)
df['bath_bed_ratio']   = df['bathrooms_fill'] / df['bedrooms_fill'].clip(1)
df['log_surface']      = np.log1p(df['surface_fill'])

# Flags de valeurs manquantes (utiles pour le modèle)
df['was_missing_surface']  = df['surface_area'].isna().astype(int)
df['was_missing_bedrooms'] = df['bedrooms'].isna().astype(int)
df['was_missing_bath']     = df['bathrooms'].isna().astype(int)

print(' Features numériques créées')

 Features numériques créées


In [7]:
# ══════════════════════════════════════════════════════════════
# 6. TARGET ENCODING (prix médian par ville)
# ══════════════════════════════════════════════════════════════
# Calculer le prix médian par ville (target encoding avec smoothing)
city_stats = df.groupby('city')['price'].agg(['median', 'mean', 'count'])
global_median = df['price'].median()
smoothing = 10  # smoothing factor

city_stats['city_price_target'] = (
    (city_stats['count'] * city_stats['median'] + smoothing * global_median) /
    (city_stats['count'] + smoothing)
)
df = df.merge(city_stats[['city_price_target']].rename(columns={'city_price_target':'city_prix_m2_median'}),
              on='city', how='left')
df['city_prix_m2_median'] = df['city_prix_m2_median'].fillna(global_median)

# Encoding catégoriel
le_type   = LabelEncoder()
le_source = LabelEncoder()
df['property_type_enc'] = le_type.fit_transform(df['property_type_clean'].fillna('Autre'))
df['source_enc']        = le_source.fit_transform(df['source'].fillna('inconnu'))

# Sauvegarder les encodeurs
import joblib
joblib.dump({'le_type': le_type, 'le_source': le_source, 'city_stats': city_stats},
            '../data/encoders.pkl')

print('✅ Target encoding et encodeurs créés')
print(city_stats.sort_values('median', ascending=False).head(10)[['median','count']].apply(
    lambda col: col.apply(lambda x: f'{x/1e6:.0f}M' if col.name == 'median' else int(x)), axis=0))

✅ Target encoding et encodeurs créés
              median  count
city                       
Diass           678M      2
Cité Batrain    500M      1
Ndayane         430M      2
Saly Portudal   300M      3
Cambérène       297M      3
Kanel           270M      9
Medina          250M     18
Grand-Dakar     250M      1
Fann            238M     10
Fass            235M      2


In [8]:
# ══════════════════════════════════════════════════════════════
# 7. VARIABLE CIBLE + SAUVEGARDES
# ══════════════════════════════════════════════════════════════
df['log_price'] = np.log1p(df['price'])

# Liste des features finales pour la modélisation
NUM_FEATURES = [
    'surface_fill', 'bedrooms_fill', 'bathrooms_fill',
    'rooms_total', 'surface_per_room', 'bath_bed_ratio', 'log_surface',
    'dist_mer', 'dist_centre', 'dist_aeroport', 'dist_port', 'dist_ucad',
    'dist_vdn', 'dist_corniche', 'dist_parc', 'dist_aibd',
    'log_dist_mer', 'log_dist_centre',
    'zone_premium', 'is_premium',
    'was_missing_surface', 'was_missing_bedrooms', 'was_missing_bath',
    'city_prix_m2_median', 'n_premium_feats',
] + list(NLP_FEATURES.keys())

CAT_FEATURES = ['property_type_clean', 'source']

# Vérifier
available = [f for f in NUM_FEATURES if f in df.columns]
missing   = [f for f in NUM_FEATURES if f not in df.columns]
print(f'Features disponibles : {len(available)}/{len(NUM_FEATURES)}')
if missing: print(f'Manquantes : {missing}')

# Sauvegarder
df.to_csv('../data/dataset_features.csv', index=False)

# Config features
features_config = {
    'NUMERIC_FEATURES':      available,
    'CATEGORICAL_FEATURES':  CAT_FEATURES,
    'TARGET':                'price',
    'LOG_TARGET':            'log_price',
    'n_features_total':      len(available) + len(CAT_FEATURES),
    'sources':               list(df['source'].unique()),
}
with open('../properties/ml/features_config.json', 'w') as f:
    json.dump(features_config, f, indent=2)

print(f'\n✅ Sauvegardé : dataset_features.csv ({len(df):,} lignes)')
print(f'✅ features_config.json : {features_config["n_features_total"]} features')

Features disponibles : 45/45

✅ Sauvegardé : dataset_features.csv (2,768 lignes)
✅ features_config.json : 47 features
